In [ ]:
# Data Acquisition - ERA5-Land 2x Daily (00:00 and 12:00)

import cdsapi
import os
import xarray as xr
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

print("Starting ERA5-Land 2x Daily Download (00:00 and 12:00)\n")

BASE_DIR = r"I:\Data Science Projects\disease_prediction"
DATA_RAW = os.path.join(BASE_DIR, "data", "raw")
CLIMATE_RAW = os.path.join(DATA_RAW, "climate")
os.makedirs(CLIMATE_RAW, exist_ok=True)

print(f"Saving to: {CLIMATE_RAW}\n")

c = cdsapi.Client()

dataset = "reanalysis-era5-land"

output_files = []

for year in tqdm(range(2010, 2025), desc="Downloading years"):
    request = {
        "variable": ["2m_temperature", "total_precipitation"],
        "year": str(year),
        "month": [f"{m:02d}" for m in range(1, 13)],
        "day": [f"{d:02d}" for d in range(1, 32)],
        "time": ["00:00", "12:00"],           # Only two times per day
        "data_format": "netcdf",
        "area": [4.5, 29.5, -1.5, 35.0]
    }
    
    output_file = os.path.join(CLIMATE_RAW, f"era5_land_uganda_{year}_2xdaily.nc")
    
    print(f"\nDownloading year {year}...")
    c.retrieve(dataset, request, output_file)
    
    size_mb = os.path.getsize(output_file) / (1024 * 1024)
    print(f"  Saved: {size_mb:.1f} MB")
    output_files.append(output_file)



In [ ]:
# Final Merge - With Proper Coordinate Handling

import xarray as xr
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

CLIMATE_RAW = r"I:\Data Science Projects\disease_prediction\data\raw\climate"

# Get all extracted data_0.nc files
extracted_files = sorted(list(Path(CLIMATE_RAW).rglob("data_0.nc")))

print(f"Found {len(extracted_files)} data files\n")

# Inspect the first file structure
if extracted_files:
    print("=== Structure of first file ===")
    ds_test = xr.open_dataset(extracted_files[0], engine="netcdf4")
    print(ds_test)
    print("\nDimensions:", list(ds_test.dims.keys()))
    print("Coordinates:", list(ds_test.coords.keys()))
    print("Variables:", list(ds_test.data_vars.keys()))

# Merge all files
print("\nMerging files...")

ds_list = []
for f in extracted_files:
    ds = xr.open_dataset(f, engine="netcdf4")
    ds_list.append(ds)

# Concatenate along the time dimension
ds_merged = xr.concat(ds_list, dim="valid_time")   # try "valid_time" first

# Rename if needed
if "valid_time" in ds_merged.coords:
    ds_merged = ds_merged.rename({"valid_time": "time"})

ds_merged = ds_merged.sortby("time")

final_file = os.path.join(CLIMATE_RAW, "era5_land_uganda_2xdaily_2010_2024_merged.nc")
ds_merged.to_netcdf(final_file)

print(f"\n Merge completed successfully!")
print(f"Final file: {final_file}")
print(f"Total time steps: {len(ds_merged.time)}")
print("Variables:", list(ds_merged.data_vars.keys()))
print("Time range:", ds_merged.time.min().values, "→", ds_merged.time.max().values)